## Bài tập:
- Dựa trên bài lab đã học ở tuần trước (CNN thuần), hãy thay đổi thành các tập dữ liệu khác như: 
    - cat and dog
    - CIFAR-10
    - PlantVillage

Sử dụng các phương pháp phân tích dữ liệu, cân bằng dữ liệu, ...

Cố gắng thay đổi các tham số sao cho độ chính xác phải lớn hơn 90% và tránh trường hợp overfitting.

Không sử dụng các mô hình re-train hoặc các mô hình như Resnet, Convnext tiny, ...



# CIFA-10

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np

device = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")

transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
])

train_dataset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform_train)
test_dataset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform_test)

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=128, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=128, shuffle=False)

classes = ('plane', 'car', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck')

  5%|▍         | 7.70M/170M [01:07<23:49, 114kB/s]   


KeyboardInterrupt: 

In [ ]:
class CIFAR10_CNN(nn.Module):
    def __init__(self):
        super(CIFAR10_CNN, self).__init__()
        self.block1 = nn.Sequential(
            nn.Conv2d(3, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Conv2d(64, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )
        self.block2 = nn.Sequential(
            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.Conv2d(128, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )
        self.block3 = nn.Sequential(
            nn.Conv2d(128, 256, 3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.Conv2d(256, 256, 3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )
        
        self.dropout = nn.Dropout(0.5)
        self.fc = nn.Sequential(
            nn.Linear(256 * 4 * 4, 1024),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.Linear(512, 10)
        )

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        return x

In [ ]:
model = CIFAR10_CNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.9, weight_decay=5e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', patience=5, factor=0.5)

In [ ]:
loss_values, accuracy_values = [], []

for epoch in range(100):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
    
    epoch_acc = 100 * correct / total
    loss_values.append(running_loss / len(train_loader))
    accuracy_values.append(epoch_acc)
    
    # Cập nhật Scheduler
    scheduler.step(epoch_acc)
    
    print(f"Epoch {epoch+1}/100 | Loss: {loss_values[-1]:.4f} | Acc: {epoch_acc:.2f}% ")

In [ ]:
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(loss_values, color='b', label='Loss')
plt.grid(True); plt.legend()
plt.subplot(1, 2, 2)
plt.plot(accuracy_values, color='g', label='Accuracy')
plt.axhline(y=90, color='r', linestyle='--') 
plt.grid(True); plt.legend()
plt.show()

In [ ]:
model.eval()
correct, total = 0, 0
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f"\n Độ chính xác trên tập test: {100 * correct / total:.2f}%")

In [ ]:

def visualize_prediction():
    model.eval()
    images, labels = next(iter(test_loader))
    images, labels = images.to(device), labels.to(device)
    
    outputs = model(images)
    _, predicted = torch.max(outputs, 1)
    
    mean, std = np.array([0.4914, 0.4822, 0.4465]), np.array([0.2023, 0.1994, 0.2010])
    
    fig, axes = plt.subplots(1, 5, figsize=(15, 4))
    for i in range(5):
        img = images[i].cpu().permute(1, 2, 0).numpy() * std + mean
        axes[i].imshow(np.clip(img, 0, 1))
        
        color = 'green' if predicted[i] == labels[i] else 'red'
        axes[i].set_title(f"Dự đoán: {classes[predicted[i]]}\nThật: {classes[labels[i]]}", color=color, fontsize=9)
        axes[i].axis('off')
    
    plt.tight_layout()
    plt.show()

visualize_prediction()

# CAT AND DOG

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader, random_split
import matplotlib.pyplot as plt
import numpy as np
import os
from PIL import Image

device = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")

class CatDogDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.image_files = [f for f in os.listdir(root_dir) if f.endswith('.jpg')]

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        img_name = self.image_files[idx]
        img_path = os.path.join(self.root_dir, img_name)
        image = Image.open(img_path).convert('RGB')
        label = 0 if 'cat' in img_name.lower() else 1
        if self.transform:
            image = self.transform(image)
        return image, label

transform_train = transforms.Compose([
    transforms.Resize((128, 128)), 
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2), 
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))
])

transform_test = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))
])

full_dataset = CatDogDataset(root_dir='./dogs-vs-cats/train', transform=transform_train)
train_size = int(0.85 * len(full_dataset)) 
test_size = len(full_dataset) - train_size
train_data, test_data = random_split(full_dataset, [train_size, test_size])

test_data.dataset.transform = transform_test

train_loader = DataLoader(train_data, batch_size=64, shuffle=True) 
test_loader = DataLoader(test_data, batch_size=64, shuffle=False)

classes = ['cat', 'dog']

In [ ]:
class CatDog_CNN(nn.Module):
    def __init__(self):
        super(CatDog_CNN, self).__init__()
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        self.relu1 = nn.ReLU()
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)
        self.relu2 = nn.ReLU()
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(128)
        self.relu3 = nn.ReLU()
        self.pool3 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.conv4 = nn.Conv2d(128, 256, kernel_size=3, padding=1)
        self.bn4 = nn.BatchNorm2d(256)
        self.relu4 = nn.ReLU()
        self.pool4 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.dropout = nn.Dropout(0.5)
        self.fc1 = nn.Linear(256 * 8 * 8, 512)
        self.fc2 = nn.Linear(512, 2)

    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu1(x)
        x = self.pool1(x)
        x = self.conv2(x)
        x = self.bn2(x)
        x = self.relu2(x)
        x = self.pool2(x)
        x = self.conv3(x)
        x = self.bn3(x)
        x = self.relu3(x)
        x = self.pool3(x)
        x = self.conv4(x)
        x = self.bn4(x)
        x = self.relu4(x)
        x = self.pool4(x)
        x = x.view(x.size(0), -1) 
        x = self.fc1(x)
        x = torch.relu(x)
        x = self.dropout(x)
        x = self.fc2(x)
        
        return x

In [ ]:
model = CatDog_CNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-2) 
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', patience=5, factor=0.5)

In [ ]:
loss_values, accuracy_values = [], []

for epoch in range(50):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
    
    epoch_acc = 100 * correct / total
    loss_values.append(running_loss / len(train_loader))
    accuracy_values.append(epoch_acc)
    
    scheduler.step(epoch_acc)
    
    print(f"Epoch {epoch+1}/50, Loss: {loss_values[-1]:.4f}, Train Acc: {epoch_acc:.2f}%")

In [ ]:
plt.figure(figsize=(15, 5))

plt.subplot(1, 2, 1)
plt.plot(range(1, len(loss_values) + 1), loss_values, color='royalblue', linewidth=2, label='Training Loss')
plt.title('Loss', fontsize=14)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Loss', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.7)
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(range(1, len(accuracy_values) + 1), accuracy_values, color='seagreen', linewidth=2, label='Training Accuracy')
plt.axhline(y=90, color='red', linestyle='--', label='Mục tiêu 90%')
plt.title('Accuracy', fontsize=14)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Accuracy (%)', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.7)
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
model.eval()
correct, total = 0, 0
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

test_acc = 100 * correct / total
print(f"\n Độ chính xác trên tập test: {test_acc:.2f}%")

In [ ]:
def visualize_prediction():
    model.eval()
    data_iter = iter(test_loader)
    images, labels = next(data_iter)
    images_gpu, labels_gpu = images.to(device), labels.to(device)
    
    with torch.no_grad():
        outputs = model(images_gpu)
        _, predicted = torch.max(outputs, 1)
    
    fig, axes = plt.subplots(1, 5, figsize=(18, 5))

    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    
    classes = ['Cat', 'Dog'] 
    
    for i in range(5):
        img = images[i].permute(1, 2, 0).numpy()
        img = img * std + mean 
        img = np.clip(img, 0, 1)
        axes[i].imshow(img)
        is_correct = predicted[i] == labels_gpu[i]
        title_color = 'green' if is_correct else 'red'
        axes[i].set_title(f"Dự đoán: {classes[predicted[i]]}\nThật: {classes[labels_gpu[i]]}", 
                          color=title_color, fontsize=11, fontweight='bold')
        axes[i].axis('off')
    
    plt.suptitle("Kết quả dự đoán mẫu trên tập Test", fontsize=16)
    plt.tight_layout()
    plt.show()

visualize_prediction()

# PLANT VILLAGE

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader, random_split
import matplotlib.pyplot as plt
import numpy as np
import os

device = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")

transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))
])

data_path = './plantvillage dataset/color' 
full_dataset = ImageFolder(root=data_path, transform=transform)

train_size = int(0.8 * len(full_dataset))
test_size = len(full_dataset) - train_size
train_data, test_data = random_split(full_dataset, [train_size, test_size])

train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
test_loader = DataLoader(test_data, batch_size=64, shuffle=False)

classes = full_dataset.classes
num_classes = len(classes)
print(f"Tìm thấy {num_classes} loại bệnh cây trồng.")

In [ ]:
class PlantCNN(nn.Module):
    def __init__(self, num_classes):
        super(PlantCNN, self).__init__()
        def conv_layer(in_f, out_f):
            return nn.Sequential(
                nn.Conv2d(in_f, out_f, kernel_size=3, padding=1),
                nn.BatchNorm2d(out_f),
                nn.ReLU(),
                nn.MaxPool2d(2, 2)
            )
        self.features = nn.Sequential(
            conv_layer(3, 32),   
            conv_layer(32, 64),  
            conv_layer(64, 128), 
            conv_layer(128, 256) 
        )
        self.classifier = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(256 * 8 * 8, 512),
            nn.ReLU(),
            nn.Linear(512, num_classes)
        )
    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        return self.classifier(x)

In [ ]:
model = PlantCNN(num_classes).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-2)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'max', patience=3, factor=0.5)

In [ ]:
loss_values, accuracy_values = [], []

for epoch in range(40):
    model.train()
    r_loss, correct, total = 0.0, 0, 0
    for imgs, lbls in train_loader:
        imgs, lbls = imgs.to(device), lbls.to(device)
        optimizer.zero_grad()
        out = model(imgs); loss = criterion(out, lbls)
        loss.backward(); optimizer.step()
        
        r_loss += loss.item(); _, pred = out.max(1); total += lbls.size(0); correct += pred.eq(lbls).sum().item()
    
    epoch_acc = 100. * correct / total
    loss_values.append(r_loss/len(train_loader))
    accuracy_values.append(epoch_acc)
    scheduler.step(epoch_acc)
    print(f"Epoch {epoch+1} | Loss: {loss_values[-1]:.4f} | Acc: {epoch_acc:.2f}%")

In [ ]:
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1); plt.plot(loss_values, 'b'); plt.title('Training Loss')
plt.subplot(1, 2, 2); plt.plot(accuracy_values, 'g'); plt.axhline(y=90, color='r', linestyle='--'); plt.title('Training Accuracy')
plt.show()

In [ ]:
model.eval()
correct, total = 0, 0
with torch.no_grad():
    for imgs, lbls in test_loader:
        imgs, lbls = imgs.to(device), lbls.to(device)
        out = model(imgs); _, pred = out.max(1); total += lbls.size(0); correct += pred.eq(lbls).sum().item()

print(f"\n Độ chính xác trên tập test: {100. * correct / total:.2f}%")

In [ ]:
def visualize_prediction():
    model.eval()
    images, labels = next(iter(test_loader))
    images, labels = images.to(device), labels.to(device)
    out = model(images); _, predicted = torch.max(out, 1)
    
    mean, std = np.array([0.485, 0.456, 0.406]), np.array([0.229, 0.224, 0.225])
    fig, axes = plt.subplots(1, 5, figsize=(20, 5))
    for i in range(5):
        img = images[i].cpu().permute(1, 2, 0).numpy() * std + mean
        axes[i].imshow(np.clip(img, 0, 1))
        color = 'green' if predicted[i] == labels[i] else 'red'
        axes[i].set_title(f"P: {classes[predicted[i]][:10]}...\nT: {classes[labels[i]][:10]}...", color=color, fontsize=8)
        axes[i].axis('off')
    plt.show()

visualize_prediction()